# VIIRS Post-processing

In [ ]:
from osgeo import gdal 
import os
import geopandas as gpd
import rasterio
import pandas as pd
from rasterio.mask import mask
import fiona
#conda create -n viirs python=3.10 numpy pandas geopandas rasterio gdal -c conda-forge

This notebook implements the **Patches Filtering Method (PFM)** based on the study by Yuan et al., (2019) to enhance the identification of nighttime lights associated with settlements in VIIRS monthly data.

#### Method Steps:
1. **Data Loading:** pre procesing data (crop area).
2. **Background Noise Filtering:** A radiance threshold is defined based on known unpopulated areas to remove spurious values.

In [ ]:
raw_path = os.getcwd()
viirs_raw_path= raw_path + r'\Download\viirs_avg_rad\raw'

# 1) crop only study area 
## after download must cut area are you interesting

In [ ]:
shapefile_path = raw_path + r'\shapefiles\cut_area.shp'
output_cut = raw_path + r'\Download\viirs_avg_rad\cut'
tif_files = [file for file in os.listdir(viirs_raw_path) if file.endswith(".tif")]
# crop area studies 
for tif_file in tif_files:
    tif_file_path = os.path.join(viirs_raw_path, tif_file)
    
    # Open the original raster file
    with fiona.open(shapefile_path, "r") as shapefile:
        shapes = [feature["geometry"] for feature in shapefile]
    with rasterio.open(tif_file_path) as src:

        out_image, out_transform = rasterio.mask.mask(src, shapes, crop=True)
        out_meta = src.meta

    out_meta.update({"driver": "GTiff",
                 "height": out_image.shape[1],
                 "width": out_image.shape[2],
                 "transform": out_transform})
    algorithm = tif_file
    output_raster_path = os.path.join(output_cut, algorithm)
    with rasterio.open(output_raster_path, "w", **out_meta) as dest:
        dest.write(out_image)

In [ ]:
# all tif after cuted 
tif_files = [file for file in os.listdir(output_cut) if file.endswith(".tif")]


## Importar archivos .tif de VIIRS 

In [ ]:

dir_out_background_denoise = raw_path + r'\Download\viirs_avg_rad\background_denoise\denoise'
archivos_en_directorio = os.listdir(output_cut)
archivos_tif = [archivo for archivo in archivos_en_directorio if archivo.lower().endswith('.tif')]
nuevo_nombre = os.path.splitext(archivos_tif[0])[0] + "_backgroud_denoise.tif"
nueva_ruta = os.path.join(dir_out_background_denoise,nuevo_nombre)
print(nueva_ruta)


## Import shapefile of control points
### Control points created from QGIS using the latest satellite image, utilizing water bodies, bare soils, forests, or sparsely populated areas to eliminate background noise.

In [ ]:
# Load control points shapefile
shp_file_path = raw_path + r'\shapefiles\point_control.shp'

shapefile = gpd.read_file(shp_file_path)
if shapefile.crs is None:
    shapefile.crs = "EPSG:4326"  # Set CRS if it's missing
    
# Explode the geometry (points) for easier processing
point_control = shapefile.explode()
print(type(point_control))
print(point_control)

# Set output path
output = raw_path + r'\Download\viirs_avg_rad\background_denoise\denoise'
dir_files = output_cut

# List all tif files in the directory
archivos_en_directorio = os.listdir(dir_files)
archivos_tif = [archivo for archivo in archivos_en_directorio if archivo.lower().endswith('.tif')]
print(archivos_tif)

import numpy as np
# Loop over each tif file
for tif in archivos_tif:
    image = os.path.join(output_cut, tif)
    new_name = os.path.splitext(tif)[0] + "_background_denoise.tif"
    new_dir = os.path.join(output, new_name)

    # Open the raster file
    viirsRasters = rasterio.open(image)
    data = [] 
    
    # Extract raster values at control point locations
    for point in point_control['geometry']:
        x = point.xy[0][0]
        y = point.xy[1][0]
        row, col = viirsRasters.index(x, y)
        viirs_value = viirsRasters.read(1)[row, col]
        data.append([x, y, viirs_value])
    
    # Create a dataframe to store results
    df = pd.DataFrame(data, columns=['Longitude', 'Latitude', 'VIIRS Value'])
    df = df.loc[df['VIIRS Value'] != 0]  # Remove zero values

    # Calculate the average VIIRS value
    avg_viirs = df['VIIRS Value'].mean()

    with rasterio.open(image) as src:
        viirs_data = src.read(1)  # Read raster data
        profile = src.profile  # Get profile for metadata

    # Set values less than or equal to the average as 0 to remove background noise
    viirs_data[viirs_data <= avg_viirs] = 0

    # Write the processed raster to a new GeoTIFF
    with rasterio.open(new_dir, 'w', **profile) as dst:
        dst.write(viirs_data, 1)